In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder

ACTION_MAP = {
    'воспользовался поиском': 'SEARCH',
    'воспользовался каталогом': 'CATALOG',
    'находится в карточке товара': 'CARD_VIEW',
    'просматривает сертификат товара': 'CARD_VIEW',
    'добавил в корзину': 'CART_VIEW',
    'находится в корзине': 'CART_VIEW',
    'оформление заказа': 'CART_VIEW',
    'подтверждение заказа': 'CART_VIEW',
    'оплата': 'CART_VIEW',
    'находится на странице акций': 'PROMO',
    'нажал применить промокод': 'PROMO',
    'рассрочка': 'PROMO',
    'распродажа': 'PROMO',
    'цены снижены': 'PROMO',
    'находится в списке статей': 'ARTICLES',
    'находится на главной странице': 'MAIN',
    'личный кабинет': 'CABINET',
    'избранное': 'CABINET',
}

CLASS_LIST = sorted(set(ACTION_MAP.values()))

def map_action(text):
    t = str(text).lower()
    for key, cls in ACTION_MAP.items():
        if key in t:
            return cls
    return 'OTHER'

def build_features(df):
    df = df.copy()
    df.sort_values(['sess_id', 'watch_tms'], inplace=True)
    df['action'] = df['goal_nm_lvl1'].apply(map_action)
    df['watch_tms'] = pd.to_datetime(df['watch_tms'])
    df['delta_sec'] = df.groupby('sess_id')['watch_tms'].diff().dt.total_seconds().fillna(0)
    df['delta_sec'] = df['delta_sec'].clip(0, 3600)
    df['step_idx'] = df.groupby('sess_id').cumcount()
    df['sess_len'] = df.groupby('sess_id')['action'].transform('size')
    df['unique_actions'] = df.groupby('sess_id')['action'].transform('nunique')
    df['sess_time'] = df.groupby('sess_id')['delta_sec'].transform('sum')
    df['time_from_start'] = df.groupby('sess_id')['delta_sec'].cumsum()
    df['time_to_end'] = df['sess_time'] - df['time_from_start']
    df['is_cart'] = df['goal_nm_lvl1'].str.contains('корзин', case=False, na=False)
    le = LabelEncoder()
    df['act_id'] = le.fit_transform(df['action'])
    feat_cols = [
        'act_id', 'delta_sec',
        'step_idx', 'sess_len', 'unique_actions',
        'sess_time', 'time_from_start', 'time_to_end'
    ]
    return df, feat_cols, le

df = pd.read_csv("../data/actions.csv", sep=";")
df_feat, feat_cols, le = build_features(df)
df_feat.head()